# Notebook 02 — Feature Engineering and Target Construction for CLD Modeling

## Goal

This notebook builds the canonical clone-level dataset used for downstream machine learning.

It performs two linked tasks:

### 1. Feature engineering
Transform raw early-stage assay measurements into clone-level model inputs.

- **Feature engineering** means converting raw measurements into variables that a model can use.
- Examples:
  - mean
  - standard deviation
  - slope
  - last observed value
  - qP proxy

### 2. Target construction
Compute late-stage clone outcomes from the raw database.

- **Target construction** means defining the values the model should predict.
- In this project, the main targets are:
  - late qP
  - qP drop
  - stability label
  - late aggregation

---

## Important principle: leakage-safe design

This notebook uses only **early-stage information** to build features.

- **Leakage** (or **target leakage**) means future information accidentally enters the feature set.
- That would make model performance look unrealistically good.

So the workflow is:

- early passages → features
- late passages → targets only

---

## Important term: qP proxy

In this notebook, productivity is represented using a **qP proxy**:

- `qP_proxy = titer / VCD`

This should be interpreted as a **relative productivity measure**
for clone comparison, not as a fully calibrated physical qP unit.

---

## Output

This notebook saves:

1. **features-only table**
2. **canonical modeling dataset**

The final dataset is used by:

- Notebook 03b — prediction engine
- Notebook 04b — decision engine

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path

scenario = "legacy"   # or "optimized"
n_clones = 5000

EARLY_START = 3
EARLY_END = 10
LATE_START = 24
LATE_END = 30
STABILITY_DROP_THRESHOLD = 0.30

DB_PATH = Path(f"../data/synthetic/raw/cld_{n_clones}clones_{scenario}.db")
OUT_DIR = Path("../data/synthetic/processed")

OUT_FEATURES = OUT_DIR / f"cld_features_v2_{n_clones}_{scenario}.csv"
OUT_DATASET = OUT_DIR / (
    f"cld_features_with_labels_qp_targets_v2_{LATE_START}_{LATE_END}_{n_clones}_{scenario}.csv"
)

print("DB:", DB_PATH)
print("Output features:", OUT_FEATURES)
print("Output dataset:", OUT_DATASET)
print("Early window:", EARLY_START, EARLY_END)
print("Late window:", LATE_START, LATE_END)
print("Stable threshold:", STABILITY_DROP_THRESHOLD)

DB: ../data/synthetic/raw/cld_5000clones_legacy.db
Output features: ../data/synthetic/processed/cld_features_v2_5000_legacy.csv
Output dataset: ../data/synthetic/processed/cld_features_with_labels_qp_targets_v2_24_30_5000_legacy.csv
Early window: 3 10
Late window: 24 30
Stable threshold: 0.3


## 1. Load raw assay data

We load assay-level data directly from the SQLite database.

This raw data will be used to build:

- early-stage features
- late-stage targets

Only early passages will be used for feature engineering.
Late passages will only be used for target construction.

In [2]:
conn = sqlite3.connect(DB_PATH)

assay = pd.read_sql_query("""
SELECT
    ar.assay_id,
    ar.assay_type,
    ar.value,
    ar.unit,
    ar.method,
    ar.batch_id,
    p.clone_id,
    p.passage_number,
    p.phase
FROM assay_result ar
JOIN passage p
  ON p.passage_id = ar.passage_id
""", conn)

print("Assay rows:", len(assay))
display(assay.head())

Assay rows: 605000


,assay_id,assay_type,value,unit,method,batch_id,clone_id,passage_number,phase
0,ASSAY_CLONE_0001_P01_titer,titer,3.611143e+00,g/L,ELISA,B_P01,CLONE_0001,1,early
1,ASSAY_CLONE_0001_P01_vcd,vcd,9.786417e+06,cells/mL,Vi-CELL,B_P01,CLONE_0001,1,early
2,ASSAY_CLONE_0001_P01_viability,viability,9.233340e+01,%,Vi-CELL,B_P01,CLONE_0001,1,early
3,ASSAY_CLONE_0001_P01_aggregation,aggregation,5.464017e+00,%,SEC-HPLC,B_P01,CLONE_0001,1,early
4,ASSAY_CLONE_0001_P02_titer,titer,3.959182e+00,g/L,ELISA,B_P02,CLONE_0001,2,early


## 2. Restrict to early window

We use only passages P3–10 for feature engineering.

This mimics real CLD decision timing:
- early passages are available at selection time
- late passages are not

In [3]:
assay_early = assay[
    (assay["passage_number"] >= EARLY_START) &
    (assay["passage_number"] <= EARLY_END)
].copy()

print("Early assay rows:", len(assay_early))
display(assay_early.head())

Early assay rows: 165000


,assay_id,assay_type,value,unit,method,batch_id,clone_id,passage_number,phase
8,ASSAY_CLONE_0001_P03_titer,titer,3.478151e+00,g/L,ELISA,B_P03,CLONE_0001,3,early
9,ASSAY_CLONE_0001_P03_vcd,vcd,1.118005e+07,cells/mL,Vi-CELL,B_P03,CLONE_0001,3,early
10,ASSAY_CLONE_0001_P03_viability,viability,9.304314e+01,%,Vi-CELL,B_P03,CLONE_0001,3,early
11,ASSAY_CLONE_0001_P03_aggregation,aggregation,5.641781e+00,%,SEC-HPLC,B_P03,CLONE_0001,3,early
12,ASSAY_CLONE_0001_P03_ddpcr_cn,ddpcr_cn,4.000000e+00,copies/cell,ddPCR,B_P03,CLONE_0001,3,early


## 3. Extract clone-level ddPCR feature

ddPCR copy number is measured once per clone early in the process.

We convert it into a single clone-level feature:

- `ddpcr_cn`

This can be useful because copy number may affect productivity and burden.

In [4]:
ddpcr = assay_early[assay_early["assay_type"] == "ddpcr_cn"].copy()
ddpcr = (
    ddpcr.groupby("clone_id", as_index=False)["value"]
    .mean()
    .rename(columns={"value": "ddpcr_cn"})
)

print("ddPCR clones:", len(ddpcr))
display(ddpcr.head())

ddPCR clones: 5000


,clone_id,ddpcr_cn
0,CLONE_0001,4.0
1,CLONE_0002,2.0
2,CLONE_0003,5.0
3,CLONE_0004,4.0
4,CLONE_0005,6.0


## 4. Build passage-level wide table

We reshape assay data into a clone × passage table.

This makes it easier to compute:

- summary statistics
- slopes
- curvature
- qP proxy

In [5]:
wide = assay_early.pivot_table(
    index=["clone_id", "passage_number"],
    columns="assay_type",
    values="value",
    aggfunc="mean"
).reset_index()

wide.columns.name = None
display(wide.head())

,clone_id,passage_number,aggregation,ddpcr_cn,titer,vcd,viability
0,CLONE_0001,3,5.641781,4.0,3.478151,1.118005e+07,93.043137
1,CLONE_0001,4,5.960226,NaN,3.646983,1.035721e+07,93.554481
2,CLONE_0001,5,6.105156,NaN,3.185749,1.158315e+07,94.415453
3,CLONE_0001,6,5.545210,NaN,3.414548,1.106252e+07,96.775947
4,CLONE_0001,7,5.314600,NaN,3.087901,1.059510e+07,96.160063


## 5. Construct qP proxy

We define an early productivity proxy:

- `qP_proxy = titer / VCD`

This is used as a relative productivity feature for clone comparison.

In [6]:
wide["qP_proxy"] = wide["titer"] / wide["vcd"].replace(0, np.nan)
display(wide[["clone_id", "passage_number", "titer", "vcd", "qP_proxy"]].head())

,clone_id,passage_number,titer,vcd,qP_proxy
0,CLONE_0001,3,3.478151,1.118005e+07,3.111032e-07
1,CLONE_0001,4,3.646983,1.035721e+07,3.521201e-07
2,CLONE_0001,5,3.185749,1.158315e+07,2.750331e-07
3,CLONE_0001,6,3.414548,1.106252e+07,3.086592e-07
4,CLONE_0001,7,3.087901,1.059510e+07,2.914461e-07


## 6. Build clone-level summary features

For each early-stage variable, we compute clone-level summaries:

- mean
- std
- min
- max
- coefficient of variation (CV)
- last observed value at P10

These summarize clone behavior within the early screen.

In [7]:
feature_rows = []

for clone_id, sub in wide.groupby("clone_id"):
    sub = sub.sort_values("passage_number").copy()

    row = {"clone_id": clone_id}

    for col in ["titer", "vcd", "viability", "aggregation", "qP_proxy"]:
        vals = sub[col].astype(float)

        row[f"{col}_mean"] = vals.mean()
        row[f"{col}_std"] = vals.std()
        row[f"{col}_min"] = vals.min()
        row[f"{col}_max"] = vals.max()
        row[f"{col}_cv"] = vals.std() / (vals.mean() + 1e-12)

        p10 = sub.loc[sub["passage_number"] == EARLY_END, col]
        row[f"{col}_p10"] = p10.iloc[0] if len(p10) > 0 else np.nan

    feature_rows.append(row)

features = pd.DataFrame(feature_rows)
print("Features shape after summaries:", features.shape)
display(features.head())

Features shape after summaries: (5000, 31)


,clone_id,titer_mean,titer_std,titer_min,titer_max,titer_cv,titer_p10,vcd_mean,vcd_std,vcd_min,...,aggregation_min,aggregation_max,aggregation_cv,aggregation_p10,qP_proxy_mean,qP_proxy_std,qP_proxy_min,qP_proxy_max,qP_proxy_cv,qP_proxy_p10
0,CLONE_0001,3.240001,0.244658,2.946003,3.646983,0.075512,3.116997,1.150399e+07,8.543827e+05,1.035721e+07,...,5.314600,6.114239,0.052090,5.821161,2.840245e-07,3.973083e-08,2.348336e-07,3.521201e-07,0.139885,2.576601e-07
1,CLONE_0002,0.890732,0.199584,0.492907,1.080938,0.224068,1.042496,1.436123e+07,8.165488e+05,1.312061e+07,...,5.734541,6.390833,0.036011,5.987830,6.264352e-08,1.626898e-08,3.299719e-08,7.945486e-08,0.259703,7.945486e-08
2,CLONE_0003,5.060018,0.403075,4.469948,5.464666,0.079659,4.596945,7.826375e+06,2.751202e+05,7.427478e+06,...,2.146441,3.079982,0.118921,2.535371,6.481282e-07,6.638545e-08,5.463864e-07,7.105375e-07,0.102426,5.463864e-07
3,CLONE_0004,1.277790,0.317717,0.715785,1.638754,0.248646,0.715785,1.763823e+07,1.792335e+06,1.498000e+07,...,0.621988,2.328517,0.316247,1.973116,7.432975e-08,2.431783e-08,3.688191e-08,1.059680e-07,0.327157,3.688191e-08
4,CLONE_0005,3.254988,0.137416,3.048111,3.434989,0.042217,3.144411,1.021871e+07,1.226798e+06,8.845936e+06,...,2.896673,3.453439,0.062166,2.896673,3.230290e-07,4.457588e-08,2.638336e-07,3.802250e-07,0.137993,2.687784e-07


## 7. Add dynamic early features

We compute dynamic features to capture early trajectory shape:

- full-window slope
- split slopes:
  - P3–6
  - P7–10
- curvature:
  - later slope − earlier slope

These features help distinguish:
- improving clones
- flattening clones
- early false positives

In [8]:
def slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    return np.polyfit(x[mask], y[mask], 1)[0]

dyn_rows = []

for clone_id, sub in wide.groupby("clone_id"):
    sub = sub.sort_values("passage_number").copy()
    row = {"clone_id": clone_id}

    for col in ["titer", "vcd", "viability", "aggregation", "qP_proxy"]:
        x_all = sub["passage_number"].values
        y_all = sub[col].values

        row[f"{col}_slope"] = slope(x_all, y_all)

        sub_36 = sub[(sub["passage_number"] >= 3) & (sub["passage_number"] <= 6)]
        sub_710 = sub[(sub["passage_number"] >= 7) & (sub["passage_number"] <= 10)]

        s36 = slope(sub_36["passage_number"].values, sub_36[col].values)
        s710 = slope(sub_710["passage_number"].values, sub_710[col].values)

        row[f"{col}_slope_3_6"] = s36
        row[f"{col}_slope_7_10"] = s710
        row[f"{col}_curvature"] = s710 - s36 if np.isfinite(s36) and np.isfinite(s710) else np.nan

    dyn_rows.append(row)

dyn = pd.DataFrame(dyn_rows)
print("Dynamic features shape:", dyn.shape)
display(dyn.head())

Dynamic features shape: (5000, 21)


,clone_id,titer_slope,titer_slope_3_6,titer_slope_7_10,titer_curvature,vcd_slope,vcd_slope_3_6,vcd_slope_7_10,vcd_curvature,viability_slope,...,viability_slope_7_10,viability_curvature,aggregation_slope,aggregation_slope_3_6,aggregation_slope_7_10,aggregation_curvature,qP_proxy_slope,qP_proxy_slope_3_6,qP_proxy_slope_7_10,qP_proxy_curvature
0,CLONE_0001,-0.078458,-0.065204,0.018496,0.083700,239414.040811,87332.859410,457311.720910,369978.861500,0.477047,...,0.689783,-0.516158,-0.016407,-0.014478,0.087957,0.102436,-1.268820e-08,-8.441910e-09,-9.485069e-09,-1.043159e-09
1,CLONE_0002,-0.013165,-0.002577,0.109134,0.111711,-26781.399104,-104023.055400,-345350.821477,-241327.766077,0.561119,...,0.734002,0.338729,0.025852,0.035556,-0.112573,-0.148130,-7.610942e-10,5.152232e-11,9.430391e-09,9.378869e-09
2,CLONE_0003,-0.154087,-0.053347,-0.186279,-0.132931,67939.466662,-104594.369948,187555.493962,292149.863910,0.183145,...,0.258466,-0.394055,-0.051422,0.146309,0.079323,-0.066986,-2.476998e-08,2.694125e-09,-3.703065e-08,-3.972477e-08
3,CLONE_0004,-0.120016,-0.072640,-0.166508,-0.093869,611794.308486,529605.086091,-194924.991134,-724530.077224,0.008635,...,-0.251305,-0.582035,0.166921,0.199086,-0.017957,-0.217043,-9.511840e-09,-7.480508e-09,-7.871628e-09,-3.911201e-10
4,CLONE_0005,-0.028627,-0.116330,-0.058116,0.058215,297328.777582,44416.065906,532801.804554,488385.738648,0.164029,...,2.033395,1.727520,-0.003871,0.000300,-0.165711,-0.166010,-1.193248e-08,-1.310922e-08,-2.204084e-08,-8.931621e-09


## 8. Merge all early-stage features

We merge:

- summary statistics
- dynamic features
- ddPCR feature

to create the final early-stage feature table.

In [9]:
features_v2 = (
    features
    .merge(dyn, on="clone_id", how="left")
    .merge(ddpcr, on="clone_id", how="left")
)

# compatibility alias for downstream notebooks
features_v2["early_mean_qp"] = features_v2["qP_proxy_mean"]

print("Final feature table shape:", features_v2.shape)
display(features_v2.head())

Final feature table shape: (5000, 53)


,clone_id,titer_mean,titer_std,titer_min,titer_max,titer_cv,titer_p10,vcd_mean,vcd_std,vcd_min,...,aggregation_slope,aggregation_slope_3_6,aggregation_slope_7_10,aggregation_curvature,qP_proxy_slope,qP_proxy_slope_3_6,qP_proxy_slope_7_10,qP_proxy_curvature,ddpcr_cn,early_mean_qp
0,CLONE_0001,3.240001,0.244658,2.946003,3.646983,0.075512,3.116997,1.150399e+07,8.543827e+05,1.035721e+07,...,-0.016407,-0.014478,0.087957,0.102436,-1.268820e-08,-8.441910e-09,-9.485069e-09,-1.043159e-09,4.0,2.840245e-07
1,CLONE_0002,0.890732,0.199584,0.492907,1.080938,0.224068,1.042496,1.436123e+07,8.165488e+05,1.312061e+07,...,0.025852,0.035556,-0.112573,-0.148130,-7.610942e-10,5.152232e-11,9.430391e-09,9.378869e-09,2.0,6.264352e-08
2,CLONE_0003,5.060018,0.403075,4.469948,5.464666,0.079659,4.596945,7.826375e+06,2.751202e+05,7.427478e+06,...,-0.051422,0.146309,0.079323,-0.066986,-2.476998e-08,2.694125e-09,-3.703065e-08,-3.972477e-08,5.0,6.481282e-07
3,CLONE_0004,1.277790,0.317717,0.715785,1.638754,0.248646,0.715785,1.763823e+07,1.792335e+06,1.498000e+07,...,0.166921,0.199086,-0.017957,-0.217043,-9.511840e-09,-7.480508e-09,-7.871628e-09,-3.911201e-10,4.0,7.432975e-08
4,CLONE_0005,3.254988,0.137416,3.048111,3.434989,0.042217,3.144411,1.021871e+07,1.226798e+06,8.845936e+06,...,-0.003871,0.000300,-0.165711,-0.166010,-1.193248e-08,-1.310922e-08,-2.204084e-08,-8.931621e-09,6.0,3.230290e-07


In [10]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
features_v2.to_csv(OUT_FEATURES, index=False)
print("Saved features:", OUT_FEATURES)

Saved features: ../data/synthetic/processed/cld_features_v2_5000_legacy.csv


## 9. Build late-stage targets from raw DB

We compute late-stage outcomes from the raw database using the late passage window.

Main targets:

- `late_mean_qp`
- `late_mean_aggregation`

Derived targets:

- `qp_drop_pct`
- `stable_label_30pct`

These targets are used only as labels, not as features.

In [11]:
def fetch_late_labels(conn, clone_list, late_start=24, late_end=30):
    """
    Compute late-stage mean qP and aggregation per clone.

    qP proxy is computed passage-wise as:
        qP = titer / vcd
    and then averaged over the late window.
    """
    if len(clone_list) == 0:
        return pd.DataFrame(
            columns=[
                "clone_id",
                "late_mean_titer",
                "late_mean_vcd",
                "late_mean_qp",
                "late_mean_aggregation",
            ]
        )

    placeholders = ",".join(["?"] * len(clone_list))

    query = f"""
    WITH late_assays AS (
        SELECT
            p.clone_id,
            p.passage_number,
            AVG(CASE WHEN ar.assay_type = 'titer' THEN ar.value END) AS titer_value,
            AVG(CASE WHEN ar.assay_type = 'vcd' THEN ar.value END) AS vcd_value,
            AVG(CASE WHEN ar.assay_type = 'aggregation' THEN ar.value END) AS aggregation_value
        FROM assay_result ar
        JOIN passage p
          ON p.passage_id = ar.passage_id
        WHERE p.passage_number BETWEEN ? AND ?
          AND p.clone_id IN ({placeholders})
        GROUP BY p.clone_id, p.passage_number
    )
    SELECT
        clone_id,
        AVG(titer_value) AS late_mean_titer,
        AVG(vcd_value) AS late_mean_vcd,
        AVG(titer_value / NULLIF(vcd_value, 0)) AS late_mean_qp,
        AVG(aggregation_value) AS late_mean_aggregation
    FROM late_assays
    GROUP BY clone_id
    """

    params = [late_start, late_end] + list(clone_list)
    return pd.read_sql_query(query, conn, params=params)

In [12]:
conn = sqlite3.connect(DB_PATH)

late_labels = fetch_late_labels(
    conn,
    features_v2["clone_id"].tolist(),
    late_start=LATE_START,
    late_end=LATE_END
)

conn.close()
display(late_labels.head())

,clone_id,late_mean_titer,late_mean_vcd,late_mean_qp,late_mean_aggregation
0,CLONE_0001,2.374156,1.407659e+07,1.685610e-07,6.532632
1,CLONE_0002,0.389273,1.635915e+07,2.327035e-08,9.320724
2,CLONE_0003,2.002406,1.139933e+07,1.766731e-07,4.421152
3,CLONE_0004,0.371628,1.703147e+07,2.185912e-08,2.296516
4,CLONE_0005,2.422141,1.275459e+07,1.898780e-07,2.565377


In [13]:
dataset = features_v2.merge(
    late_labels,
    on="clone_id",
    how="left"
)

dataset["early_mean_qp"] = dataset["qP_proxy_mean"]

dataset["qp_drop_pct"] = (
    (dataset["early_mean_qp"] - dataset["late_mean_qp"])
    / (dataset["early_mean_qp"] + 1e-12)
).clip(lower=0.0, upper=1.5)

dataset["stable_label_30pct"] = (
    dataset["qp_drop_pct"] <= STABILITY_DROP_THRESHOLD
).astype(int)

display(dataset.head())

,clone_id,titer_mean,titer_std,titer_min,titer_max,titer_cv,titer_p10,vcd_mean,vcd_std,vcd_min,...,qP_proxy_slope_7_10,qP_proxy_curvature,ddpcr_cn,early_mean_qp,late_mean_titer,late_mean_vcd,late_mean_qp,late_mean_aggregation,qp_drop_pct,stable_label_30pct
0,CLONE_0001,3.240001,0.244658,2.946003,3.646983,0.075512,3.116997,1.150399e+07,8.543827e+05,1.035721e+07,...,-9.485069e-09,-1.043159e-09,4.0,2.840245e-07,2.374156,1.407659e+07,1.685610e-07,6.532632,0.406525,0
1,CLONE_0002,0.890732,0.199584,0.492907,1.080938,0.224068,1.042496,1.436123e+07,8.165488e+05,1.312061e+07,...,9.430391e-09,9.378869e-09,2.0,6.264352e-08,0.389273,1.635915e+07,2.327035e-08,9.320724,0.628517,0
2,CLONE_0003,5.060018,0.403075,4.469948,5.464666,0.079659,4.596945,7.826375e+06,2.751202e+05,7.427478e+06,...,-3.703065e-08,-3.972477e-08,5.0,6.481282e-07,2.002406,1.139933e+07,1.766731e-07,4.421152,0.727409,0
3,CLONE_0004,1.277790,0.317717,0.715785,1.638754,0.248646,0.715785,1.763823e+07,1.792335e+06,1.498000e+07,...,-7.871628e-09,-3.911201e-10,4.0,7.432975e-08,0.371628,1.703147e+07,2.185912e-08,2.296516,0.705908,0
4,CLONE_0005,3.254988,0.137416,3.048111,3.434989,0.042217,3.144411,1.021871e+07,1.226798e+06,8.845936e+06,...,-2.204084e-08,-8.931621e-09,6.0,3.230290e-07,2.422141,1.275459e+07,1.898780e-07,2.565377,0.412194,0


In [14]:
dataset[
    [
        "early_mean_qp",
        "late_mean_qp",
        "qp_drop_pct",
        "stable_label_30pct",
        "late_mean_aggregation",
    ]
].isna().mean()

early_mean_qp            0.0
late_mean_qp             0.0
qp_drop_pct              0.0
stable_label_30pct       0.0
late_mean_aggregation    0.0
dtype: float64

In [15]:
dataset[
    [
        "early_mean_qp",
        "late_mean_qp",
        "qp_drop_pct",
        "stable_label_30pct",
        "late_mean_aggregation",
    ]
].describe()

,early_mean_qp,late_mean_qp,qp_drop_pct,stable_label_30pct,late_mean_aggregation
count,5.000000e+03,5.000000e+03,5000.000000,5000.000000,5000.000000
mean,4.002844e-07,1.769910e-07,0.434182,0.255400,6.160825
std,4.827627e-06,1.221427e-06,0.207136,0.436129,3.281593
min,4.445550e-09,1.241834e-09,0.000000,0.000000,0.000000
25%,8.091674e-08,4.245736e-08,0.295652,0.000000,3.599848
50%,1.450972e-07,7.959254e-08,0.439589,0.000000,5.920209
75%,2.759564e-07,1.538407e-07,0.573584,1.000000,8.379098
max,3.050702e-04,7.851137e-05,0.988908,1.000000,19.307697


## 10. Enforce canonical qP-based schema

To keep downstream notebooks stable, we remove obsolete titer-based columns if they exist.

- **Canonical schema** means the standard column set expected by later notebooks.
- This prevents mismatch between old and new naming systems.

In [16]:
legacy_cols = ["late_mean_titer", "late_mean_vcd", "productivity_drop_pct"]
to_drop = [c for c in legacy_cols if c in dataset.columns]

if to_drop:
    print("Removing legacy columns before save:", to_drop)
    dataset = dataset.drop(columns=to_drop)

print("\nSchema check after cleanup:")
for c in [
    "early_mean_qp",
    "late_mean_qp",
    "qp_drop_pct",
    "stable_label_30pct",
    "late_mean_aggregation",
    "late_mean_titer",
    "late_mean_vcd",
    "productivity_drop_pct",
]:
    print(f"{c}: {c in dataset.columns}")

Removing legacy columns before save: ['late_mean_titer', 'late_mean_vcd']

Schema check after cleanup:
early_mean_qp: True
late_mean_qp: True
qp_drop_pct: True
stable_label_30pct: True
late_mean_aggregation: True
late_mean_titer: False
late_mean_vcd: False
productivity_drop_pct: False


In [17]:
dataset.to_csv(OUT_DATASET, index=False)
print("Saved dataset:", OUT_DATASET)
print("Rows:", dataset.shape[0], "Columns:", dataset.shape[1])

Saved dataset: ../data/synthetic/processed/cld_features_with_labels_qp_targets_v2_24_30_5000_legacy.csv
Rows: 5000 Columns: 57


## Output of Notebook 02

This notebook produces two outputs:

### 1. Early-stage feature table
Clone-level features from early passages only.

### 2. Canonical modeling dataset
A leakage-safe dataset containing:

- early-stage features
- late-stage qP target
- stability target
- late-stage aggregation target

---

## Downstream usage

This output is used by:

- **Notebook 03b** — prediction engine
- **Notebook 04b** — decision engine

---

## Pipeline meaning

- Notebook 01 = understand the synthetic data
- Notebook 02 = define X and y
- Notebook 03 = predict late outcomes
- Notebook 04 = make clone selection decisions

In [18]:
print(dataset.shape)

print(
    dataset[
        ["early_mean_qp", "late_mean_qp", "qp_drop_pct", "stable_label_30pct", "late_mean_aggregation"]
    ].head()
)

print(
    dataset[
        ["qp_drop_pct", "stable_label_30pct", "late_mean_qp", "late_mean_aggregation"]
    ].describe()
)

(5000, 57)
   early_mean_qp  late_mean_qp  qp_drop_pct  stable_label_30pct  \
0   2.840245e-07  1.685610e-07     0.406525                   0   
1   6.264352e-08  2.327035e-08     0.628517                   0   
2   6.481282e-07  1.766731e-07     0.727409                   0   
3   7.432975e-08  2.185912e-08     0.705908                   0   
4   3.230290e-07  1.898780e-07     0.412194                   0   

   late_mean_aggregation  
0               6.532632  
1               9.320724  
2               4.421152  
3               2.296516  
4               2.565377  
       qp_drop_pct  stable_label_30pct  late_mean_qp  late_mean_aggregation
count  5000.000000         5000.000000  5.000000e+03            5000.000000
mean      0.434182            0.255400  1.769910e-07               6.160825
std       0.207136            0.436129  1.221427e-06               3.281593
min       0.000000            0.000000  1.241834e-09               0.000000
25%       0.295652            0.000000  4.24